<a href="https://colab.research.google.com/github/SultanKus/Eticaret/blob/main/eticaret_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Örnek resim/çizim verileri
data = {
    "id": [1, 2, 3, 4],
    "title": ["Gece Yansıması (Orijinal)", "Soyut Düşler (Tuval Baskı)", "Portre Çalışması - 01", "Suluboya Botanik Serisi"],
    "category": ["Orijinal Tuval", "Baskı", "Portre", "Suluboya"],
    "price": [1250.0, 450.0, 900.0, 350.0],
    "stock": [1, 10, 1, 5],
    "description": ["Akrilik boya, 50x70 cm tual üzerine çalışılmıştır. Tek nüshadır.",
                    "Yüksek kaliteli mat kuşe kağıda sınırlı sayıdaki imzalı baskı.",
                    "Karakalem ve kömür tozu karışık teknik portre çalışması.",
                    "Özel suluboya kağıdı üzerine el yapımı botanik illüstrasyon."],
    "image_url": [
        "https://images.unsplash.com/photo-1579783902614-a3fb3927b675?auto=format&fit=crop&w=600&q=80",
        "https://images.unsplash.com/photo-1541701494587-cb58502866ab?auto=format&fit=crop&w=600&q=80",
        "https://images.unsplash.com/photo-1579783900882-c0d3dad7b119?auto=format&fit=crop&w=600&q=80",
        "https://images.unsplash.com/photo-1513364776144-60967b0f800f?auto=format&fit=crop&w=600&q=80"
    ]
}

df = pd.DataFrame(data)
df.to_csv("products.csv", index=False)
print("Ürün veri seti başarıyla oluşturuldu!")
df.head()

Ürün veri seti başarıyla oluşturuldu!


,id,title,category,price,stock,description,image_url
0,1,Gece Yansıması (Orijinal),Orijinal Tuval,1250.0,1,"Akrilik boya, 50x70 cm tual üzerine çalışılmış...",https://images.unsplash.com/photo-157978390261...
1,2,Soyut Düşler (Tuval Baskı),Baskı,450.0,10,Yüksek kaliteli mat kuşe kağıda sınırlı sayıda...,https://images.unsplash.com/photo-154170149458...
2,3,Portre Çalışması - 01,Portre,900.0,1,Karakalem ve kömür tozu karışık teknik portre ...,https://images.unsplash.com/photo-157978390088...
3,4,Suluboya Botanik Serisi,Suluboya,350.0,5,Özel suluboya kağıdı üzerine el yapımı botanik...,https://images.unsplash.com/photo-151336477614...


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd

# Sayfa Ayarları
st.set_page_config(page_title="Sanat Atölyesi Vitrini", page_icon="🎨", layout="wide")

# Verileri Yükle
@st.cache_data
def load_data():
    return pd.read_csv("products.csv")

df = load_data()

# Başlık ve Karşılama
st.title("🎨 Sanat & İllüstrasyon Vitrini")
st.markdown("Özel çizimler, orijinal tablolar ve sınırlı sayıdaki baskılar.")

# Sidebar (Kenar Çubuğu) - Filtreler ve Sepet
st.sidebar.header("🔍 Filtreler")
selected_category = st.sidebar.selectbox("Kategori Seçin", ["Tümü"] + list(df["category"].unique()))

if selected_category != "Tümü":
    filtered_df = df[df["category"] == selected_category]
else:
    filtered_df = df

# Sepet Yönetimi (Session State)
if "cart" not in st.session_state:
    st.session_state.cart = []

st.sidebar.divider()
st.sidebar.subheader("🛒 Sepetim")
if len(st.session_state.cart) > 0:
    total_price = 0
    for item in st.session_state.cart:
        st.sidebar.write(f"- {item['title']} ({item['price']} TL)")
        total_price += item['price']
    st.sidebar.markdown(f"**Toplam Tutar: {total_price} TL**")

    if st.sidebar.button("🗑️ Sepeti Temizle"):
        st.session_state.cart = []
        st.rerun()

    shipping_name = st.sidebar.text_input("Ad Soyad")
    shipping_address = st.sidebar.text_area("Teslimat Adresi")

    if st.sidebar.button("📲 Siparişi WhatsApp ile Tamamla"):
        if shipping_name and shipping_address:
            order_details = f"Merhaba, yeni bir sipariş vermek istiyorum:\n\n*Müşteri:* {shipping_name}\n*Adres:* {shipping_address}\n\n*Ürünler:*\n"
            for item in st.session_state.cart:
                order_details += f"- {item['title']} ({item['price']} TL)\n"
            order_details += f"\nToplam: {total_price} TL"

            # WhatsApp linki oluşturma (Telefon numarasını arkadaşının numarasıyla değiştirebilirsin)
            import urllib.parse
            encoded_message = urllib.parse.quote(order_details)
            whatsapp_url = f"https://wa.me/905000000000?text={encoded_message}"

            st.sidebar.markdown(f"[📲 Siparişi Göndermek İçin Tıklayın]({whatsapp_url})", unsafe_allow_html=True)
        else:
            st.sidebar.warning("Lütfen ad soyad ve adres bilgilerini doldurun.")
else:
    st.sidebar.write("Sepetiniz henüz boş.")

# Ürünleri Kartlar Halinde Listeleme
st.divider()
cols = st.columns(2)

for index, row in filtered_df.iterrows():
    with cols[index % 2]:
        st.image(row["image_url"], use_container_width=True)
        st.subheader(row["title"])
        st.markdown(f"**Kategori:** {row['category']}")
        st.markdown(f"**Fiyat:** {row['price']} TL")
        st.write(row["description"])

        if st.button(f"Sepete Ekle", key=f"btn_{row['id']}"):
            st.session_state.cart.append({
                "id": row["id"],
                "title": row["title"],
                "price": row["price"]
            })
            st.success(f"'{row['title']}' sepete eklendi!")

Writing app.py


In [3]:
# 1. Gerekli kütüphaneleri yükleyelim
!pip install streamlit pandas

# 2. Ürün veri setini oluşturalım
import pandas as pd

data = {
    "id": [1, 2, 3, 4],
    "title": ["Gece Yansıması (Orijinal)", "Soyut Düşler (Tuval Baskı)", "Portre Çalışması - 01", "Suluboya Botanik Serisi"],
    "category": ["Orijinal Tuval", "Baskı", "Portre", "Suluboya"],
    "price": [1250.0, 450.0, 900.0, 350.0],
    "stock": [1, 10, 1, 5],
    "description": ["Akrilik boya, 50x70 cm tual üzerine çalışılmıştır. Tek nüshadır.",
                    "Yüksek kaliteli mat kuşe kağıda sınırlı sayıdaki imzalı baskı.",
                    "Karakalem ve kömür tozu karışık teknik portre çalışması.",
                    "Özel suluboya kağıdı üzerine el yapımı botanik illüstrasyon."],
    "image_url": [
        "https://images.unsplash.com/photo-1579783902614-a3fb3927b675?auto=format&fit=crop&w=600&q=80",
        "https://images.unsplash.com/photo-1541701494587-cb58502866ab?auto=format&fit=crop&w=600&q=80",
        "https://images.unsplash.com/photo-1579783900882-c0d3dad7b119?auto=format&fit=crop&w=600&q=80",
        "https://images.unsplash.com/photo-1513364776144-60967b0f800f?auto=format&fit=crop&w=600&q=80"
    ]
}
pd.DataFrame(data).to_csv("products.csv", index=False)

# 3. Streamlit app.py dosyasını oluşturalım
app_code = """
import streamlit as st
import pandas as pd

st.set_page_config(page_title="Sanat Atölyesi Vitrini", page_icon="🎨", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("products.csv")

df = load_data()

st.title("🎨 Sanat & İllüstrasyon Vitrini")
st.markdown("Özel çizimler, orijinal tablolar ve sınırlı sayıdaki baskılar.")

st.sidebar.header("🔍 Filtreler")
selected_category = st.sidebar.selectbox("Kategori Seçin", ["Tümü"] + list(df["category"].unique()))

if selected_category != "Tümü":
    filtered_df = df[df["category"] == selected_category]
else:
    filtered_df = df

if "cart" not in st.session_state:
    st.session_state.cart = []

st.sidebar.divider()
st.sidebar.subheader("🛒 Sepetim")
if len(st.session_state.cart) > 0:
    total_price = 0
    for item in st.session_state.cart:
        st.sidebar.write(f"- {item['title']} ({item['price']} TL)")
        total_price += item['price']
    st.sidebar.markdown(f"**Toplam Tutar: {total_price} TL**")

    if st.sidebar.button("🗑️ Sepeti Temizle"):
        st.session_state.cart = []
        st.rerun()

    shipping_name = st.sidebar.text_input("Ad Soyad")
    shipping_address = st.sidebar.text_area("Teslimat Adresi")

    if st.sidebar.button("📲 Siparişi WhatsApp ile Tamamla"):
        if shipping_name and shipping_address:
            order_details = f"Merhaba, yeni bir sipariş vermek istiyorum:\\n\\n*Müşteri:* {shipping_name}\\n*Adres:* {shipping_address}\\n\\n*Ürünler:*\\n"
            for item in st.session_state.cart:
                order_details += f"- {item['title']} ({item['price']} TL)\\n"
            order_details += f"\\nToplam: {total_price} TL"

            import urllib.parse
            encoded_message = urllib.parse.quote(order_details)
            whatsapp_url = f"https://wa.me/905000000000?text={encoded_message}"

            st.sidebar.markdown(f"[📲 Siparişi Göndermek İçin Tıklayın]({whatsapp_url})", unsafe_allow_html=True)
        else:
            st.sidebar.warning("Lütfen ad soyad ve adres bilgilerini doldurun.")
else:
    st.sidebar.write("Sepetiniz henüz boş.")

st.divider()
cols = st.columns(2)

for index, row in filtered_df.iterrows():
    with cols[index % 2]:
        st.image(row["image_url"], use_container_width=True)
        st.subheader(row["title"])
        st.markdown(f"**Kategori:** {row['category']}")
        st.markdown(f"**Fiyat:** {row['price']} TL")
        st.write(row["description"])

        if st.button(f"Sepete Ekle", key=f"btn_{row['id']}"):
            st.session_state.cart.append({
                "id": row["id"],
                "title": row["title"],
                "price": row["price"]
            })
            st.success(f"'{row['title']}' sepete eklendi!")
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Dosyalar hazır! Şimdi Streamlit'i başlatabiliriz.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 97.6 MB/s eta 0:00:00
Dosyalar hazır! Şimdi Streamlit'i başlatabiliriz.


In [4]:
# Streamlit'i dış bağlantı ile çalıştırma (LocalTunnel kullanarak)
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-09-22 14:39:45.418 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.69.190.84:8501

  Stopping...
^C
